In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-04-29 04:15:16,989 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-29 04:15:16,990 INFO sqlalchemy.engine.Engine COMMIT
✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [ ]:
# TODO: Create additional 10 articles

# created additional 10 articles in cultpass_articles.jsonl


In [7]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [8]:
cultpass_articles

[{'title': 'How to Reserve a Spot for an Event',
  'content': 'If a user asks how to reserve an event:\n\n- Guide them to the CultPass app\n- Instruct them to browse the experience catalog and tap \'Reserve\'\n- If it\'s a premium or limited event, check if reservation confirmation is required via email\n- Remind them to arrive at least 15 minutes early with their QR code visible\n\n**Suggested phrasing:**\n"You can reserve an experience by opening the CultPass app, selecting your desired event, and tapping \'Reserve\'. Be sure to arrive 15 minutes early with your QR code ready."',
  'tags': 'reservation, events, booking, attendance'},
 {'title': "What's Included in a CultPass Subscription",
  'content': 'Each user is entitled to 4 cultural experiences per month, which may include:\n- Art exhibitions\n- Museum entries\n- Music concerts\n- Film screenings and more\n\nSome premium experiences may require an additional fee (visible in the app).\n\n**Suggested phrasing:**\n"Your CultPass s

In [9]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [10]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [11]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [12]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [13]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


# Tests

In [14]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [15]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='c5395f93-316e-412f-9819-e31dc0fce7ca', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='b5b6d7cc-a689-481a-97df-5308f13f9ae3', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='f6b00c0c-387f-4d8c-b41d-1e192d9b1d5a', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='2ff7ca09-a11b-4ce2-b3f9-aa1369887ab0', title='How to Handle Login Issues?')>
<Knowledge(article_id='99a7dff9-f4d9-4cd5-a779-9468f28a6738', title='Understanding Membership Tiers')>
<Knowledge(article_id='b6e60689-3bd4-42eb-ad17-493ac857e643', title='What to Do If an Event Is Fully Booked')>
<Knowledge(article_id='fdcc2862-e5bf-448b-a809-2aa74e790916', title='How to Check Your Monthly Quota')>
<Knowledge(article_id='d1009016-b34d-4bb7-a291-7f685007d57f', title='Late Arrival Policy for Events')>
<Knowledge(article_id='6edd97ac-3bdc-4627-b678-31caa645ba9f', title='How QR Code Entry Works')>
<Knowledge(article_id='978f0a7e-2274-4d2f-9eb6-6699

In [16]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='555758ec-6224-4967-b66c-793c9c3ec1f5', user_name='Alice Kingsley', external_user_id='a4ab87')>


In [17]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='68485bff-6790-45a4-b996-a2244c194826', role='user', content='I can't log in to my Cultpass ...')>


In [18]:

import sqlite3

conn = sqlite3.connect("data/core/udahub.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS ticket_logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ticket_id TEXT NOT NULL,
    stage TEXT NOT NULL,
    payload TEXT,
    created_at TEXT
)
""")

conn.commit()
conn.close()

print("✅ ticket_logs table created")


✅ ticket_logs table created


In [22]:

import sqlite3

conn = sqlite3.connect("data/core/udahub.db")
cur = conn.cursor()

cur.execute("SELECT ticket_id, stage, payload FROM ticket_logs ORDER BY id DESC LIMIT 10")
rows = cur.fetchall()
conn.close()

rows


[('MEM-DEMO-02',
  'analysis_kb_success',
  '{"confidence": 0.7999999999999999, "intent": "account_action", "sources": ["How to Cancel or Pause a Subscription", "How to Check Your Monthly Quota", "What\'s Included in a CultPass Subscription"], "history_used": 6}'),
 ('MEM-DEMO-02',
  'analysis_start',
  '{"intent": "account_action", "ticket_metadata": {"urgency": "low", "complexity": "low", "status": "open", "issue_type": "general"}, "tools_used": ["retrieve_articles"], "history_loaded": 6}'),
 ('MEM-DEMO-02',
  'analysis_account_success',
  '{"confidence": 0.95, "escalated": false, "user_id": "a4ab87"}'),
 ('MEM-DEMO-02',
  'analysis_start',
  '{"intent": "account_action", "ticket_metadata": {"urgency": "low", "complexity": "low", "status": "open", "issue_type": "general"}, "tools_used": ["retrieve_articles", "lookup_account"], "history_loaded": 6}'),
 ('MEM-DEMO-02',
  'analysis_account_failed',
  '{"confidence": 0.4, "escalated": true, "error": "No user found for this email (after P